# Preprocess RNA Embeddings #

In that nootbook, we'll preprocess RNA embeddings which extractec using two pre-trained language models: (1) ERNIE-RNA, and (2) RiNALMo. For more information about the methods, refer to:

(1) ERNIE-RNA:

- paper: https://www.nature.com/articles/s41467-025-64972-0
- github: https://github.com/Bruce-ywj/ERNIE-RNA

(2) RiNALMo:

- paper: https://www.nature.com/articles/s41467-025-60872-5
- github: https://github.com/lbcb-sci/RiNALMo

In [179]:
import os
import h5py
import pickle
import pandas as pd
import numpy as np
import torch

In [180]:
# we used that file to extract embeddings for both methods; order to the RNA is the same; no duplicated sequences (only unique)
RNA_SEQ_PATH = '/home/gavrilev/Thesis/src/saved_objects/rna_sequences.txt'

with open(RNA_SEQ_PATH, 'r') as f:
    sequences = [line.strip() for line in f if line.strip()]

In [175]:
sequences[6] # original seqeuces (without dealing with nucleotides 'T', 'X' and 'Y')

'CGCGAATTCGCGTTTTCGCGAATTCGCG'

In [176]:
sequences = [seq.replace("T", "U").replace("Y", "U").replace("X", "U") for seq in sequences]
sequences[6] # after dealing with invalid nucleotides (without dealing with nucleotides 'T', 'X' and 'Y')

'CGCGAAUUCGCGUUUUCGCGAAUUCGCG'

In [181]:
seq2emb = {}
for seq in sequences:
    #cleaned_seq = seq.replace("T", "U").replace("Y", "U").replace("X", "U")
    seq2emb[seq] = {'ernie': [], 'rinalmo': []}

We'll create two RNA embedding `.h5` files: (1) one **without padding** the RNA sequences to the maximum sequence length in our pool of unique RNA sequences. This version is used when training with `batch_size = 1`, where padding is unnecessary, and
(2) one **with padding** the sequences to the maximum sequence length, which enables training with larger batch sizes.

In [182]:
# with padding with 0 vectors to max RNA sequence length
with h5py.File("rna_embeddings.h5", "w") as f:
    for seq, data in seq2emb.items():
        grp = f.create_group(seq)
        grp.create_dataset("ernie", data=np.array(data['ernie']), compression="gzip")
        grp.create_dataset("rinalmo", data=np.array(data['rinalmo']), compression="gzip")

# full RNA embeddings, without padding to max RNA sequence length
with h5py.File("rna_embeddings_full.h5", "w") as f:
    for seq, data in seq2emb.items():
        grp = f.create_group(seq)
        grp.create_dataset("ernie", data=np.array(data['ernie']), compression="gzip")
        grp.create_dataset("rinalmo", data=np.array(data['rinalmo']), compression="gzip")

## Preprocess ERNIE-RNA Embeddings ##

In [238]:
cls_embeddings_path = '/home/gavrilev/Thesis/Transfer Learning: PT Models/ERNIE-RNA/results/ernie_rna_representations/test_seqs/cls_embedding.npy'
all_embeddings_path = '/home/gavrilev/Thesis/Transfer Learning: PT Models/ERNIE-RNA/results/ernie_rna_representations/test_seqs/all_embedding.npy'

In [239]:
#cls_embeddings = np.load(cls_embeddings_path) # sequence level embeddings
all_embeddings = np.load(all_embeddings_path, mmap_mode='r') # nucleotide-level embedding (matrix per RNA)

all_embeddings.shape

(295, 12, 2924, 768)

In [240]:
# we'll use the last transformer layer (i.e., 12)
last_layer = all_embeddings[:, -1, :, :]
last_layer.shape

(295, 2924, 768)

In [241]:
seventh_rna_emb = last_layer[6, :, :] # 7
seventh_rna_seq = sequences[6] # 7 

print(f'seventh RNN -> {seventh_rna_seq}, len -> {len(seventh_rna_seq)}')
seventh_rna_emb[:31]

seventh RNN -> CGCGAATTCGCGTTTTCGCGAATTCGCG, len -> 28


memmap([[-0.25182411, -0.08716655,  0.23306064, ...,  0.16479406,
         -0.61737937,  0.17419429],
        [-0.3469643 , -0.33284411,  0.29032335, ..., -0.42809135,
         -0.44722623, -0.824772  ],
        [-0.7582944 ,  0.28758615,  0.36651951, ..., -0.35241804,
         -0.4514907 , -0.56877279],
        ...,
        [-0.78531235,  0.02072765,  0.34169129, ..., -0.34422156,
         -0.51761031, -0.59620965],
        [-0.58647364, -0.11087679, -0.28412998, ..., -0.08844274,
         -0.40703526, -0.33386263],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ]])

the seventh sequence length is 28, and we got embeddings size of (2924, 768) where the first row and the last corresponds to CLS & EOS tokens.

They add +2 because the ERNIE-RNA model uses two extra tokens for every sequence: a [CLS] token at the start and an [EOS] token at the end. Therefore, every processed sequence has len(sequence) + 2 tokens.

In [242]:
# check for the longest RNA sequence
idx = sequences.index(max(sequences, key=len))

longest_rna_emb = last_layer[idx, :, :] 
longest_rna_seq = sequences[idx] # len(lonest_seq) -> 2922 

print(f'seventh RNA -> {longest_rna_seq}, len -> {len(longest_rna_seq)}')
longest_rna_emb.shape # 2922 + 2 for CLS & EOS

seventh RNA -> UUGGCUACUAUGCCAGCUGGUGGAUUGCUCGGCUCAGGCGCUGAUGAAGGACGUGCCAAGCUGCGAUAAGCCAUGGGGAGCCGCACGGAGGCGAAGAACCAUGGAUUUCCGAAUGAGAAUCUCUCUAACAAUUGCUUCGCGCAAUGAGGAACCCCGAGAACUGAAACAUCUCAGUAUCGGGAGGAACAGAAAACGCAAUGUGAUGUCGUUAGUAACCGCGAGUGAACGCGAUACAGCCCAAACCGAAGCCCUCACGGGCAAUGUGGUGUCAGGGCUACCUCUCAUCAGCCGACCGUCUCGACGAAGUCUCUUGGAACAGAGCGUGAUACAGGGUGACAACCCCGUACUCGAGACCAGUACGACGUGCGGUAGUGCCAGAGUAGCGGGGGUUGGAUAUCCCUCGCGAAUAACGCAGGCAUCGACUGCGAAGGCUAAACACAACCUGAGACCGAUAGUGAACAAGUAGUGUGAACGAACGCUGCAAAGUACCCUCAGAAGGGAGGCGAAAUAGAGCAUGAAAUCAGUUGGCGAUCGAGCGACAGGGCAUACAAGGUCCCUCGACGAAUGACCGACGCGCGAGCGUCCAGUAAGACUCACGGGAAGCCGAUGUUCUGUCGUACGUUUUGAAAAACGAGCCAGGGAGUGUGUCUGCAUGGCAAGUCUAACCGGAGUAUCCGGGGAGGCACAGGGAAACCGACAUGGCCGCAGGGCUUUGCCCGAGGGCCGCCGUCUUCAAGGGCGGGGAGCCAUGUGGACACGACCCGAAUCCGGACGAUCUACGCAUGGACAAGAUGAAGCGUGCCGAAAGGCACGUGGAAGUCUGUUAGAGUUGGUGUCCUACAAUACCCUCUCGUGAUCUAUGUGUAGGGGUGAAAGGCCCAUCGAGUCCGGCAACAGCUGGUUCCAAUCGAAACAUGUCGAAGCAUGACCUCCGCCGAGGUAGUCUGUGAGGUAGAGCGACCGAUUGGUGUGUCCGCCUCCGAGA

(2924, 768)

Next, we extract the exact sequence nucleotide level embeddings from ERNIE-RNA generated embeddings tensor, and then we pad to max length with zeros

In [243]:
ernie_dict_padded = {}      # rna_id -> Tensor (max_seq_len, 768
ernie_dict = {}  # without padding embeddings to max sequence length
seq_len_dict = {}     # rna_id -> true nucleotide length (before padding)
max_seq_len = len(max(sequences ,key=len))

for i, rna in enumerate(seq2emb.keys()):
    seq = sequences[i]
    L = len(seq)

    # slice out nucleotide embeddings: positions 1..L (drop CLS/EOS)
    nuc_emb = torch.from_numpy(all_embeddings[i][0][1:L+1]).float()  # (L, 768)
    ernie_dict[rna] = nuc_emb # without padding
    
    # truncate if longer than max_seq_len
    if L > max_seq_len:
        nuc_emb = nuc_emb[:max_seq_len]
        L = max_seq_len

    # pad with zeros if shorter
    if L < max_seq_len:
        pad = torch.zeros(max_seq_len - L, 768, dtype=nuc_emb.dtype)
        nuc_emb = torch.cat([nuc_emb, pad], dim=0)

    # sanity check
    assert nuc_emb.shape == (max_seq_len, 768), f"Bad shape for {rna}: {nuc_emb.shape}"

    ernie_dict_padded[rna] = nuc_emb
    seq_len_dict[rna] = L

In [244]:
random_sequence_1 = sequences[200]
random_sequence_2 = sequences[176]

len(random_sequence_1), len(random_sequence_2)

(281, 24)

In [245]:
def test_seq_shapes(seq, seq_lenth):
    print(f'#'*80)
    print(f'current sequence length -> {seq_lenth}')
    print(f'with padding -> {ernie_dict_padded[seq].shape}')
    print(ernie_dict_padded[seq][:seq_lenth+2]) # padded to max sequence length (2922)
    print(f'#'*80)
    print(f'without padding -> {ernie_dict[seq].shape}')
    print(ernie_dict[seq]) # without padding
    print(f'#'*80)

In [246]:
test_seq_shapes(random_sequence_1, len(random_sequence_1)) # from nucleotide number 13, padded to max sequence length (2922)

################################################################################
current sequence length -> 281
with padding -> torch.Size([2922, 768])
tensor([[ 0.4871,  0.3169,  0.3280,  ..., -0.3122, -0.1221, -0.0528],
        [-0.6973, -0.4550, -0.6953,  ..., -0.1346,  0.0585, -0.1254],
        [-1.2711, -0.9480, -1.2469,  ..., -0.0221,  0.0948, -0.2472],
        ...,
        [ 0.3834, -0.0612, -0.4893,  ..., -0.0751, -0.3363, -0.1640],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])
################################################################################
without padding -> torch.Size([281, 768])
tensor([[ 0.4871,  0.3169,  0.3280,  ..., -0.3122, -0.1221, -0.0528],
        [-0.6973, -0.4550, -0.6953,  ..., -0.1346,  0.0585, -0.1254],
        [-1.2711, -0.9480, -1.2469,  ..., -0.0221,  0.0948, -0.2472],
        ...,
        [-0.9655, -0.6603, -0.8939,  ..., -0.1801, -0.0188,  0.0550]

## Preprocess RiNALMo Embeddings ##

In [247]:
#from rinalmo.pretrained import get_pretrained_model
#import flash_attn
#import torch
#from rinalmo.pretrained import get_pretrained_model

#DEVICE = "cuda:0"

# Pipeline for extracting RiNALMo embeddings & creating dictionary
# that maps every RNA sequence -> RiNALMo embedding (not padded)

'''

model, alphabet = get_pretrained_model(model_name="giga-v1")
model = model.to(device=DEVICE)
model.eval()

tokens = torch.tensor(alphabet.batch_tokenize(seqs), dtype=torch.int64, device=DEVICE)
with torch.no_grad(), torch.cuda.amp.autocast():
  outputs = model(tokens)
'''

'''
BATCH_SIZE = 5
DEVICE = "cuda"

# in every batch its padds to the longest sequence length and add 2 for start&end tokens
# so in order to extract per nucleotide embeddings we'll take [1:len(seq)+1]
def chunked(iterable, n):
    """Yield successive n-sized chunks."""
    for i in range(0, len(iterable), n):
        yield iterable[i:i+n]

all_embeddings = []

for batch in chunked(sequences, BATCH_SIZE):

    tokens_np = alphabet.batch_tokenize(batch)   # numpy array (batch, max_len)
    
    tokens = torch.tensor(tokens_np, dtype=torch.int64, device=DEVICE)
    
    with torch.no_grad(), torch.cuda.amp.autocast():
        outputs = model(tokens)    
    
    batch_repr = outputs["representation"].float().cpu().numpy()
    all_embeddings.extend(batch_repr)

'''


## Preprocess batched RNA sequence embeddings & remove first and last tokens + padded tokens
'''
seq2emb = {}

for i, emb in enumerate(all_embeddings):
    seq2emb[sequences[i]] = emb[1:len(sequences[i])+1] # remove first token, last & all padded tokens
'''

### Save RiNALMo embeddings; path:
### r'/home/gavrilev/Thesis/Transfer Learning: PT Models/RiNALMo/rinalmo_embeddings.pkl'
'''
with open("rinalmo_embeddings.pkl", "wb") as f:
    pickle.dump(seq2emb, f, protocol=pickle.HIGHEST_PROTOCOL)

'''

'\nwith open("rinalmo_embeddings.pkl", "wb") as f:\n    pickle.dump(seq2emb, f, protocol=pickle.HIGHEST_PROTOCOL)\n\n'

In [248]:
rinalmo_dict_padded = {}     # rna_seq -> Tensor (max_seq_len, 1280)
rinalmo_dict = {} # without padding to max sequence length with zero vectors
seq_len_dict = {}      # rna_seq -> true nucleotide length

In [249]:
rinalmo_emb_path = r'/home/gavrilev/Thesis/Transfer Learning: PT Models/RiNALMo/rinalmo_embeddings.pkl'
with open(rinalmo_emb_path, "rb") as f:
    rinalmo_emb_dict = pickle.load(f)

In [250]:
random_seq = sequences[10]
len(random_seq)

67

In [251]:
print(f'RNA with sequence length -> {len(random_seq)}, emb shape -> {rinalmo_emb_dict[random_seq].shape}')

RNA with sequence length -> 67, emb shape -> (67, 1280)


In [252]:
'''
Here, we create dictionary that maps each RNA sequence (same order as in 'sequences' list)
to its corresponding RiNALMo embedding, which padded to max sequence length (i.e., 2922)
with zero vectors
'''

for rna_seq, emb in rinalmo_emb_dict.items():
    # emb shape: (L, 1280)
    if not torch.is_tensor(emb):
        emb = torch.from_numpy(emb).float()

    L = emb.size(0)
    rinalmo_dict[rna_seq] = emb # not padded
    seq_len_dict[rna_seq] = min(L, max_seq_len)
    
    # truncate if needed
    if L > max_seq_len:
        emb = emb[:max_seq_len]

    # pad if needed
    if L < max_seq_len:
        pad = torch.zeros(max_seq_len - L, emb.size(1), dtype=emb.dtype)
        emb = torch.cat([emb, pad], dim=0)

    # final safety check
    assert emb.shape == (max_seq_len, 1280), f"Bad shape for RNA length {L}"

    rinalmo_dict_padded[rna_seq] = emb # padded to max sequence length (i.e., 2922)

In [253]:
rinalmo_dict_padded[random_seq].shape # padded to -> 2922

torch.Size([2922, 1280])

In [254]:
# senaty checks:
print(f'RNA with sequence length -> {len(random_seq)}, emb shape -> {rinalmo_dict_padded[random_seq].shape}')
print(f'#'*80)
print(rinalmo_dict_padded[random_seq][:69]) 

'''
from nucleotide number 67, all rows in the embeddings matrix are padded to 1280 with zeros
'''

RNA with sequence length -> 67, emb shape -> torch.Size([2922, 1280])
################################################################################
tensor([[ 0.9134, -0.4069, -1.0648,  ..., -0.6716,  0.9097,  0.6519],
        [ 0.2249,  1.1469, -0.0032,  ..., -0.1657,  1.5316, -0.3688],
        [-0.6059, -0.6879, -1.5992,  ..., -0.3002,  0.8584, -0.2758],
        ...,
        [-0.3885, -0.5751, -0.2185,  ..., -0.3737,  1.1794, -0.2219],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])


'\nfrom nucleotide number 67, all rows in the embeddings matrix are padded to 1280 with zeros\n'

## Save Embeddings into h5d File ##

### Senaty Checks ###

In [255]:
seq1, seq2 = sequences[77], sequences[199]
lseq1, lseq2 = len(seq1), len(seq2)
lseq1, lseq2

(18, 161)

In [256]:
ernie_dict_padded[seq1][0:20], ernie_dict[seq1] # len(seq) = 18; can see last two rows padded in the first results

(tensor([[ 0.5221,  0.7094,  0.3564,  ..., -0.1913, -0.0804,  0.3870],
         [-0.8399, -0.2311, -0.8153,  ..., -0.0930,  0.1725,  0.3571],
         [-1.6298, -1.1881, -1.3462,  ..., -0.0577,  0.1658,  0.0960],
         ...,
         [ 1.3255,  0.7327,  0.3525,  ..., -0.1414,  0.0636, -0.0254],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]),
 tensor([[ 0.5221,  0.7094,  0.3564,  ..., -0.1913, -0.0804,  0.3870],
         [-0.8399, -0.2311, -0.8153,  ..., -0.0930,  0.1725,  0.3571],
         [-1.6298, -1.1881, -1.3462,  ..., -0.0577,  0.1658,  0.0960],
         ...,
         [-1.2888, -1.2919, -1.3386,  ..., -0.0931,  0.0626,  0.0516],
         [ 0.1618, -0.6135, -0.8899,  ..., -0.0650, -0.0719, -0.0560],
         [ 1.3255,  0.7327,  0.3525,  ..., -0.1414,  0.0636, -0.0254]]))

In [257]:
# check that index=161 (length+1) is padded in rna with sequence length of 161
assert torch.equal(ernie_dict_padded[seq2][161], torch.zeros(768))

In [258]:
rinalmo_dict_padded[seq1][0:20], rinalmo_dict[seq1] # len(seq) = 18; can see last two rows padded in the first results

(tensor([[ 0.2681,  1.1840,  0.0156,  ..., -0.7134, -0.2232,  0.7241],
         [ 0.6008,  1.2743, -0.0299,  ..., -0.7049, -0.1325, -0.3540],
         [ 1.8741, -1.0188, -0.7972,  ..., -0.3072, -0.7112,  0.4555],
         ...,
         [ 1.2124, -1.6687, -1.1791,  ..., -0.9555, -0.7214,  1.1296],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]),
 tensor([[ 0.2681,  1.1840,  0.0156,  ..., -0.7134, -0.2232,  0.7241],
         [ 0.6008,  1.2743, -0.0299,  ..., -0.7049, -0.1325, -0.3540],
         [ 1.8741, -1.0188, -0.7972,  ..., -0.3072, -0.7112,  0.4555],
         ...,
         [ 0.4628,  0.7556,  0.8441,  ..., -1.7304, -0.5530,  0.5169],
         [ 0.6434,  0.0172,  0.3717,  ..., -1.1290,  0.7016,  1.1832],
         [ 1.2124, -1.6687, -1.1791,  ..., -0.9555, -0.7214,  1.1296]]))

In [259]:
len(rinalmo_dict_padded.keys())

295

In [ ]:
def generate_h5_rna_emb_file(rna_h5_emb_path, erine_emb_dict, rinalmo_emb_dict):
    count_seq = 0
    with h5py.File(rna_h5_emb_path, "w") as f:
        for rna_seq in sequences:
            
            # senaty checks that the first padded vector is zero vector, and the last one isnt - thus valid
            assert not torch.equal(ernie_tensor[rna_seq][len(rna_seq)-1], torch.zeros(768))
            assert not torch.equal(rinalmo_padded[rna_seq][len(rna_seq)-1], torch.zeros(1280))
            if len(rna_seq) != max_seq_len:
                assert torch.equal(ernie_tensor[rna_seq][len(rna_seq)], torch.zeros(768))
                assert torch.equal(rinalmo_padded[rna_seq][len(rna_seq)], torch.zeros(1280))
    
            cleaned_rna_seq = rna_seq.replace('T', 'U').replace('Y', 'U').replace('X', 'U')
            if cleaned_rna_seq in f:
                print(cleaned_rna_seq)
                continue
            count_seq += 1
            grp = f.create_group(cleaned_rna_seq)
            grp.create_dataset("ernie", data=ernie_tensor[rna_seq].numpy(), compression="gzip", compression_opts=4) # (max_seq_len, 768) 
            grp.create_dataset("rinalmo", data=rinalmo_padded[rna_seq].numpy(), compression="gzip", compression_opts=4) # (max_seq_len, 1280)
            grp.create_dataset("seq_len", data=np.array(seq_len_dict[rna_seq], dtype=np.int32))
    print(count_seq) # should be 294  

In [268]:
seq_with_x_or_y = [seq for seq in sequences if 'X' in seq or 'Y' in seq] # sequence with invalide nucleotides 'X'/'Y'
seq_with_t = [seq for seq in sequences if 'T' in seq] # sequences with nucleotide 'T', which usually present in DNA, not RNA
# replace nucleotide 'T' (present in DNA) with 'U', since RNA uses uracil instead of thymine
len(seq_with_x_or_y), len(seq_with_t)

(2, 6)

In [270]:
'''
Padded Version:

Here, we generate h5 file that maps each unique preprocessed RNA after we replace all
invalid nucleotide (i.e., 'T', 'X' and 'Y') with 'U', to its corresponding RiNALMo
and ERNIE-RNA embeddings metrix (nucleotide-level embeddigs; shape max_seq_len x 768/1280, 
where max_seq_len is the longest sequence in our dataset, 768 for ERNIE-RNA and 1280 for RiNALMo)

Note: there are only 2 unique RNA sequences
with 'X'/'Y'), and 'T' usually present in DNA, whereas in RNA there are usually 'U'
instead of 'T' (6 unique RNAs with nucleotide 'T')
'''
rna_emb_path = '/home/gavrilev/Thesis/src/embeddings/rna/rna_embeddings_padded.h5' # ernie-rna & rinalmo embeddigns, padded to max sequence length (i.e., 2922)
count_seq = 0
with h5py.File(rna_emb_path, "w") as f:
    for rna_seq in sequences:
        
        # senaty checks that the first padded vector is zero vector, and the last one isnt - thus valid
        assert not torch.equal(ernie_dict_padded[rna_seq][len(rna_seq)-1], torch.zeros(768))
        assert not torch.equal(rinalmo_dict_padded[rna_seq][len(rna_seq)-1], torch.zeros(1280))
        if len(rna_seq) != max_seq_len:
            assert torch.equal(ernie_dict_padded[rna_seq][len(rna_seq)], torch.zeros(768))
            assert torch.equal(rinalmo_dict_padded[rna_seq][len(rna_seq)], torch.zeros(1280))

        cleaned_rna_seq = rna_seq.replace('T', 'U').replace('Y', 'U').replace('X', 'U')
        if cleaned_rna_seq in f:
            print(cleaned_rna_seq)
            continue
        count_seq += 1
        grp = f.create_group(cleaned_rna_seq) # append cleaned RNA sequence version;
        grp.create_dataset("ernie", data=ernie_dict_padded[rna_seq].numpy(), compression="gzip", compression_opts=4) # add original embeddings (max_seq_len, 768) 
        grp.create_dataset("rinalmo", data=rinalmo_dict_padded[rna_seq].numpy(), compression="gzip", compression_opts=4) # # add original embeddings (max_seq_len, 1280)
        grp.create_dataset("seq_len", data=np.array(seq_len_dict[rna_seq], dtype=np.int32))
print(count_seq) # should be 294

CUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUG
294


In [271]:
'''
Not Padded Version:

Here, we generate h5 file that maps each unique preprocessed RNA after we replace all
invalid nucleotide (i.e., 'T', 'X' and 'Y') with 'U', to its corresponding RiNALMo
and ERNIE-RNA embeddings metrix (nucleotide-level embeddigs; shape L x 768/1280, where
L is sequence length, 768 for ERNIE-RNA and 1280 for RiNALMo)

Note: there are only 2 unique RNA sequences
with 'X'/'Y'), and 'T' usually present in DNA, whereas in RNA there are usually 'U'
instead of 'T' (6 unique RNAs with nucleotide 'T')
'''
rna_emb_path = '/home/gavrilev/Thesis/src/embeddings/rna/rna_embeddings_clean.h5' # ernie-rna & rinalmo embeddigns, not padded 
count_seq = 0
with h5py.File(rna_emb_path, "w") as f:
    for rna_seq in sequences:
        cleaned_rna_seq = rna_seq.replace('T', 'U').replace('Y', 'U').replace('X', 'U')
        if cleaned_rna_seq in f:
            print(cleaned_rna_seq)
            continue
        count_seq += 1
        grp = f.create_group(cleaned_rna_seq) # append cleaned RNA sequence version;
        grp.create_dataset("ernie", data=ernie_dict[rna_seq].numpy(), compression="gzip", compression_opts=4) # add original embeddings (L, 768) 
        grp.create_dataset("rinalmo", data=rinalmo_dict[rna_seq].numpy(), compression="gzip", compression_opts=4) # # add original embeddings (L, 1280)
        grp.create_dataset("seq_len", data=np.array(seq_len_dict[rna_seq], dtype=np.int32))
print(count_seq) # should be 294

CUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUGCUG
294


*NOTES*: 
* There was single unique RNA sequence which after preprocessing in (replacing all nucleotides 'T' -> 'U'), we got RNA sequence which
already present in our sequences pool. Therefore, number of sequence in each h5 file will be 295, rather than 294
* We added to each `.h5` file the preprocessed version of each RNA to match AlphaFold3 sequences (which also preprocessed the same way), in order
  to get the correct embedding in the __getitem___(...) method of `MolRNADataset()` object
* We used the original embeddings (not preprocessed) of every pre-trained methods, that because they take care of invalid nucleotides in their methods, and we want to keep the original embedding.

In [272]:
# senaty checks 
rna_emb_path_clean = '/home/gavrilev/Thesis/src/embeddings/rna/rna_embeddings_clean.h5' # ernie-rna & rinalmo embeddigns, not padded 
rna_emb_path_padd = '/home/gavrilev/Thesis/src/embeddings/rna/rna_embeddings_padded.h5' # ernie-rna & rinalmo embeddigns, padded to max sequence length (i.e., 2922)

rna_emb_padd = h5py.File(rna_emb_path_padd, "r")
rna_emb_clean = h5py.File(rna_emb_path_clean, "r")

In [281]:
seq = sequences[6] # 'CGCGAATTCGCGTTTTCGCGAATTCGCG' -> with invalid nucleotide 'T'
clean_seq = seq.replace('T', 'U').replace('Y', 'U').replace('X', 'U')

In [285]:
#rna_emb = rna_emb_padd[seq] # KeyError: "Unable to synchronously open object (object 'CGCGAATTCGCGTTTTCGCGAATTCGCG' doesn't exist)"
rna_emb = rna_emb_padd[clean_seq]
len(clean_seq)

28

In [286]:
rna_emb["ernie"][:30] # from nucleotide 28 all rows padded to zero vectors

array([[ 0.51888454,  0.54597175,  0.3363045 , ..., -0.15999953,
        -0.0944246 ,  0.38418296],
       [-0.98627496, -0.7896303 , -0.7697799 , ..., -0.09491271,
         0.09982937,  0.20730872],
       [-1.5919396 , -1.1128196 , -1.6259354 , ..., -0.02981452,
         0.17364962,  0.13173243],
       ...,
       [-1.6542389 , -1.3686203 , -0.46322453, ..., -0.1619722 ,
         0.11843461,  0.3129303 ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], dtype=float32)

In [287]:
rna_emb["rinalmo"][:30] # from nucleotide 28 all rows padded to zero vectors

array([[ 1.7659276 ,  0.03180379,  0.48223263, ..., -0.22048946,
         1.2009581 ,  1.1341898 ],
       [ 1.6034927 , -1.0707963 , -1.6218811 , ...,  0.49218038,
        -0.72102714,  1.0019934 ],
       [ 1.2508552 ,  0.3814416 ,  0.86148936, ...,  0.27344686,
         0.6603246 ,  0.9737521 ],
       ...,
       [ 1.2561094 , -1.1803745 , -0.8851346 , ..., -0.2782849 ,
         0.29939735,  0.61674523],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], dtype=float32)

In [288]:
rna_emb_cleab = rna_emb_clean[clean_seq]

In [289]:
rna_emb_cleab["ernie"] # shape is equal to |L| (i.e., 28)

<HDF5 dataset "ernie": shape (28, 768), type "<f4">

In [290]:
rna_emb_cleab["rinalmo"] # shape is equal to |L| (i.e., 28)

<HDF5 dataset "rinalmo": shape (28, 1280), type "<f4">

In [292]:
rna_emb_cleab["ernie"][:], rna_emb_cleab["rinalmo"][:]

(array([[ 0.51888454,  0.54597175,  0.3363045 , ..., -0.15999953,
         -0.0944246 ,  0.38418296],
        [-0.98627496, -0.7896303 , -0.7697799 , ..., -0.09491271,
          0.09982937,  0.20730872],
        [-1.5919396 , -1.1128196 , -1.6259354 , ..., -0.02981452,
          0.17364962,  0.13173243],
        ...,
        [ 0.25157902,  1.13509   ,  1.4013362 , ..., -0.1652775 ,
          0.08475121,  0.11957788],
        [-1.1365429 ,  0.07487889,  0.73376757, ..., -0.20577322,
          0.13867368,  0.34888846],
        [-1.6542389 , -1.3686203 , -0.46322453, ..., -0.1619722 ,
          0.11843461,  0.3129303 ]], dtype=float32),
 array([[ 1.7659276 ,  0.03180379,  0.48223263, ..., -0.22048946,
          1.2009581 ,  1.1341898 ],
        [ 1.6034927 , -1.0707963 , -1.6218811 , ...,  0.49218038,
         -0.72102714,  1.0019934 ],
        [ 1.2508552 ,  0.3814416 ,  0.86148936, ...,  0.27344686,
          0.6603246 ,  0.9737521 ],
        ...,
        [ 1.6015401 , -1.3835828 , -1.0